# Langbridge SDK Local Runtime Notebook

This notebook uses `LangbridgeClient.local(config_path=...)` against the local runtime defined in `langbridge_config.yml`.

In [ ]:
from pathlib import Path
import sys

EXAMPLE_DIR = Path.cwd()
REPO_ROOT = EXAMPLE_DIR.parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import pandas as pd
from langbridge import LangbridgeClient
from setup import setup_database

CONFIG_PATH = EXAMPLE_DIR / "langbridge_config.yml"
setup_database()
client = LangbridgeClient.local(config_path=str(CONFIG_PATH))

## List datasets

In [ ]:
datasets = await client.datasets.list()
pd.DataFrame([item.model_dump(mode="json") for item in datasets.items])

## Query a semantic dataset

In [ ]:
country_sales = await client.datasets.query(
    "shopify_orders",
    metrics=["net_sales", "gross_margin"],
    dimensions=["country"],
    order={"net_sales": "desc"},
    limit=5,
)
pd.DataFrame(country_sales.rows)

## Run SQL directly

In [ ]:
sql_result = client.sql.query(
    query="""
    SELECT acquisition_channel, ROUND(SUM(net_revenue), 2) AS net_sales
    FROM orders_enriched
    GROUP BY acquisition_channel
    ORDER BY net_sales DESC
    LIMIT 5
    """,
)
pd.DataFrame(sql_result.rows)

## Ask the configured local agent

In [ ]:
answer = await client.agents.ask("Show me top countries by net sales this quarter")
answer.text, pd.DataFrame(answer.result["rows"])